In [1]:
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

class AgentState(TypedDict):
    state_message: str
    question: str
    Supervisor: bool = False
    messages: Annotated[Sequence[BaseMessage], add_messages]
    database_schemas: str
    next_step: str
    query: str
    columns: list[str]
    max_num_retries_debug: int
    num_retries_debug_sql: int
    result_debug_sql: str
    result_debug_bi: str
    error_msg_debug_sql: str
    df_data: list
    df_shape: tuple
    df_columns: list
    df_dtypes: dict
    explanation_query: str
    visualization_request: str
    python_code_data_visualization: str
    python_code_validated: bool  
    num_retries_debug_python_code_data_visualization: int
    result_debug_python_code_data_visualization: str 
    error_msg_debug_python_code_data_visualization: str

class SupervisorAgentResponse(BaseModel):
    response: str = Field(description="Your response to the user's question in Portuguese-Brazilian")
    next_step: str = Field(description="The next step to be executed or END")

class SQLQueryResponse(BaseModel):
    explain: str = Field(description="The explanation of the query with how columns were used and the logic of the query")
    query: str = Field(description="The SQL query ready to be executed")

class ReactAgentResponse(BaseModel):
    response: str = Field(description="Resposta final clara e direta para o usuário, em Português-Brasileiro.")
    link_sources: str = Field(description="Breve resumo das fontes utilizadas, incluindo links ou deixe vazio caso não tenha usado fontes externas.")

class AgentBiExpertResponse(BaseModel):
    response: str = Field(description="The response to the text, chart or table to be generated in Portuguese-Brazilian")
    next_step: str = Field(description="The next step to be executed or END")

class PythonCodeDataVisualizationResponse(BaseModel):
    explain: str = Field(description="The explanation of the python code for data visualization")
    python_code_data_visualization: str = Field(description="The python code for data visualization")

In [3]:
from langgraph.graph import END

def route_supervisor(state: AgentState) -> AgentState:
    if state['next_step'] == 'search_tables_and_schemas':
        return 'Search_Tables_and_Schemas'
    else:
        return END

def route_search_tables_and_schemas(state: AgentState) -> AgentState:
    if state["next_step"] == "redis_with_cache":
        # state["database_error"] = 1
        return "Redis_With_Cache"
    else:
        return END

def route_with_cache(state: AgentState) -> AgentState:
    # if state["messages"][-1].content == "Resposta não encontrada na cache":
    if state["next_step"] == "Agent_SQL_Validator":
        return "Agent_SQL_Validator"
    else:
        return "Agent_SQL_Writer"

def route_sql_validator(state: AgentState) -> str:
    """
    Determina se deve continuar para BI expert ou repetir validação SQL.
    """
    if (state['result_debug_sql'] == "Pass" or 
        state.get('num_retries_debug_sql', 0) >= state.get('max_num_retries_debug', 3)):
        return 'Agent_BI'
    else:
        return 'Agent_SQL_Writer'


def route_to_bi(state: AgentState) -> AgentState:
    if state['result_debug_sql']=="Pass" or state.get('num_retries_debug_sql', 0) >= state.get('max_num_retries_debug', 3):
        return 'Agent_BI'
    else:
        return 'Agent_SQL_Validator'

def route_to_python_code(state: AgentState) -> AgentState:
    if state['next_step'] == 'agent_python_code_data_visualization_generator_node':
        return 'Agent_Python_Generator'
    else:
            return 'Supervisor_Agent'

def route_python_validator(state: AgentState) -> str:
    """
    Determina se deve terminar o workflow ou repetir validação Python.
    """
    if (state['result_debug_python_code_data_visualization'] == "Pass" or 
        state.get('num_retries_debug_python_code_data_visualization', 0) >= state.get('max_num_retries_debug', 3)):
        return "Supervisor_Agent"
    else:
        return "Agent_Python_Generator"

def route_agent_tools(state: AgentState) -> AgentState:
    """
    Determina se deve continuar para o próximo nó ou responder o usuário
    """
    if state['next_step'] == 'agent_tools':
        return 'Agent_Tools'
    else:
        return END